In [25]:
import numpy as np 
import pandas as pd
from dataclasses import dataclass ## for defining datatypes remove the need for defining classes with init functions ##
from typing import List, Dict ## for defining datatypes for lists and dictionaries ##


 ### defining datatypes  ###
@dataclass
class Job:
    id: int
    workload: float
 
@dataclass 
class Node:
    id: int
    speed: float
    cost_per_hour: float
    
    
    

In [26]:

    
def generate_cloud_environment(num_jobs=50):
    nodes = [
        Node(id=1, speed=100, cost_per_hour=0.02),
        Node(id=2, speed=250, cost_per_hour=0.08),
        Node(id=3, speed=500, cost_per_hour=0.15)
    ]

    np.random.seed(42)
    mu, sigma = 6, 1.2  ## the mean and standard deviation for workload ##

    workloads = np.random.lognormal(mean=mu, sigma=sigma, size=num_jobs)  ## generating workloads using lognormal distribution ##
    workload = np.clip(workloads, a_min=10, a_max=None) 

    jobs = [Job(id=i, workload=workload[i]) for i in range(num_jobs)] 

    return jobs, nodes 


jobs, nodes = generate_cloud_environment(num_jobs=10)

print("Nodes:")
for node in nodes:
    print(f"Node ID: {node.id}, Speed: {node.speed} MIPS, Cost per Hour: ${node.cost_per_hour}")

print("\nJobs:")    
for job in jobs:
    print(f"Job ID: {job.id}, Workload: {job.workload:.2f} MI")

Nodes:
Node ID: 1, Speed: 100 MIPS, Cost per Hour: $0.02
Node ID: 2, Speed: 250 MIPS, Cost per Hour: $0.08
Node ID: 3, Speed: 500 MIPS, Cost per Hour: $0.15

Jobs:
Job ID: 0, Workload: 732.20 MI
Job ID: 1, Workload: 341.75 MI
Job ID: 2, Workload: 877.63 MI
Job ID: 3, Workload: 2508.99 MI
Job ID: 4, Workload: 304.61 MI
Job ID: 5, Workload: 304.61 MI
Job ID: 6, Workload: 2683.98 MI
Job ID: 7, Workload: 1013.25 MI
Job ID: 8, Workload: 229.67 MI
Job ID: 9, Workload: 773.61 MI


In [34]:
def evaluate_schedule(jobs: List[Job], nodes: List[Node], assignments: List[int]) -> dict:
    """
    Takes a 1D array of assignments and returns the makespan and cost.
    Example assignments: [1, 3, 2] means:
    - job 0 is assigned to node 1
    - job 1 is assigned to node 3
    - job 2 is assigned to node 2

    Args:
        jobs: List of Job objects.
        nodes: List of Node objects.
        assignments: List of node IDs corresponding to each job's assigned node.

    Returns:
        A dictionary with schedule metrics.
    """
    # 1. Create empty queues for each node (using the Node's actual ID)
    node_queues = {node.id: [] for node in nodes}

    # 2. Route the jobs to their assigned nodes
    for job_idx, assigned_node_id in enumerate(assignments):
        current_job = jobs[job_idx]
        node_queues[assigned_node_id].append(current_job)

    total_cost = 0.0
    node_completion_times = {}

    # 3. Process each node's queue
    for node in nodes:
        queue = node_queues[node.id]

        # --- THE LOCAL HEURISTIC (Shortest Job First) ---
        # Sort the queue locally so the smallest workloads run first
        queue.sort(key=lambda j: j.workload)

        node_active_time = 0.0

        # Execute the jobs
        for job in queue:
            execution_time = job.workload / node.speed
            node_active_time += execution_time

        # Calculate cost for this specific node (Convert seconds to hours)
        node_cost = (node_active_time / 3600) * node.cost_per_hour
        total_cost += node_cost

        # Record how long this node took to finish all its work
        node_completion_times[node.id] = node_active_time

    # 4. The Makespan is the time of the node that finishes absolutely last
    makespan = max(node_completion_times.values()) if node_completion_times else 0.0

    return {
        "makespan_seconds": makespan,
        "total_cost_dollars": total_cost,
        "node_details": node_completion_times
    }

In [35]:
# Extract the valid node IDs (1, 2, and 3 from our previous cell)
available_node_ids = [n.id for n in nodes]

# Generate a random schedule (The "Chromosome")
# e.g., randomly assigning each of the 10 jobs to Node 1, 2, or 3
np.random.seed(42) # Keeping it reproducible
random_schedule = np.random.choice(available_node_ids, size=len(jobs))

print(f"Generated Random Schedule (Array):")
print(random_schedule)
print("-" * 40)

# Run the random schedule through our engine
results = evaluate_schedule(jobs, nodes, random_schedule)

print("--- Simulation Results (Random Baseline) ---")
print(f"Makespan (Total Time):   {results['makespan_seconds']:.2f} seconds")
print(f"Total Cloud Cost:        ${results['total_cost_dollars']:.4f}")
print("\nIndividual Node Completion Times:")
for node_id, time_spent in results['node_details'].items():
    print(f"  Node {node_id}: {time_spent:.2f} seconds")

Generated Random Schedule (Array):
[3 1 3 3 1 1 3 2 3 3]
----------------------------------------
--- Simulation Results (Random Baseline) ---
Makespan (Total Time):   15.61 seconds
Total Cloud Cost:        $0.0008

Individual Node Completion Times:
  Node 1: 9.51 seconds
  Node 2: 4.05 seconds
  Node 3: 15.61 seconds
